# Servidor de Inferência LLM - Manejo Inteligente de Pragas
Este notebook configura o ambiente remoto (Google Colab - GPU T4) para atuar como o servidor de inferência do sistema distribuído.

*** Executa o modelo Llama 3 e expor uma API REST temporária via túnel HTTP (ngrok) para o Orquestrador local.

### 1. Instalação do Ollama

In [ ]:
!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

### 2.Inicialização do Servidor

In [ ]:
import subprocess
import time
import os

# Configurações de rede essenciais para rodar via tunel
ambiente = os.environ.copy()
ambiente["OLLAMA_ORIGINS"] = "*"              # Libera o CORS para requisições externas
ambiente["OLLAMA_HOST"] = "0.0.0.0:11434"     # Escuta em todas as interfaces, não só localhost

# Inicia o Ollama em background passando as variáveis de ambiente
subprocess.Popen(["ollama", "serve"], env=ambiente)
print("Servidor Ollama iniciando na porta 11434 (0.0.0.0)...")
time.sleep(5)
print("Servidor rodando!")

### 3.Instalando do Modelo Llama 3

In [ ]:
!ollama pull llama3.1

### 4.Instalando ngrok

In [ ]:
!pip install pyngrok

### 5.Criação do Túnel ngrok

In [ ]:
from pyngrok import ngrok, conf
from google.colab import userdata

try:
    NGROK_TOKEN = userdata.get("NGROK_TOKEN")
    conf.get_default().auth_token = NGROK_TOKEN
except userdata.SecretNotFoundError:
    print("Token do ngrok não encontrado!")
    print("Vá em: Colab > Secrets (🔑) > Adicione NGROK_TOKEN com seu token do ngrok")
else:
    # Cria o túnel na porta padrão do Ollama (11434)
    listener = ngrok.connect(11434)
    public_url = listener.public_url

    print("\n[INFO] Túnel ngrok estabelecido com sucesso.")
    print(f"[INFO] OLLAMA_BASE_URL (.env): {public_url}")
    print(f"[INFO] Endpoint de chat:        {public_url}/api/chat\n")

### 6. Teste de Validação

In [ ]:
import requests

# Usa a URL gerada dinamicamente pelo ngrok na célula anterior
url = f"{public_url}/api/chat"

headers = {
    "ngrok-skip-browser-warning": "true"
}

dados = {
    "model": "llama3.1",
    "messages": [
        {"role": "system", "content": "Você é um assistente agrícola. Responda de forma concisa."},
        {"role": "user", "content": "Responda em uma frase: por que a chuva atrapalha a aplicação de defensivos agrícolas?"}
    ],
    "stream": False
}

print(f"Enviando requisição para: {url}")
resposta = requests.post(url, headers=headers, json=dados)

if resposta.status_code == 200:
    print("\n--- RESPOSTA ---")
    print(resposta.json()["message"]["content"])
else:
    print(f"\n[ERRO HTTP {resposta.status_code}] Algo deu errado!")
    print(resposta.text)